In [1]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [2]:
%%writefile kernel.cu
__global__ void myKernel(float* a, float*b, float*c, int n){
    // thread - threadIdx.x
    // block - blockIdx.x
    // block size - blockDim.x

    // elementwise Add

    // first find thread index
    int index = blockIdx.x * blockDim.x + threadIdx.x;
    if(index < n){
        c[index] = a[index] + b[index];
    }

}

Overwriting kernel.cu


In [3]:
%%writefile main.cu

#include "kernel.cu"
#include <stdio.h>
#include <chrono>

int main() {
  // declare in cpu
  int n = 1000;
  float h_a[1000];
  float h_b[1000];
  float h_c[1000];
  for(int i = 0; i < n ;i++){
    h_a[i] = i;
    h_b[i] = n-i;
    h_c[i] = 0;
  }

  // declare for gpu
  float* d_a;
  cudaMalloc(&d_a, n * sizeof(float));
  float* d_b;
  cudaMalloc(&d_b, n * sizeof(float));
  float* d_c;
  cudaMalloc(&d_c, n * sizeof(float));

  // put cpu values in gpu
  auto start = std::chrono::high_resolution_clock::now();
  cudaMemcpy(d_a, h_a, n*sizeof(float), cudaMemcpyHostToDevice);
  cudaMemcpy(d_b, h_b, n*sizeof(float), cudaMemcpyHostToDevice);
  auto end = std::chrono::high_resolution_clock::now();
  auto duration = std::chrono::duration_cast<std::chrono::microseconds>(end - start);
  printf("copying 2 %d size values %lld\n", n, (long long)duration.count());


  // call kernel
  int threadPerBlock = 256;
  int numBlocks = ((n+threadPerBlock-1)/threadPerBlock);
  // timing
  start = std::chrono::high_resolution_clock::now();
  myKernel<<<numBlocks, threadPerBlock>>>(d_a, d_b, d_c, n);
  cudaDeviceSynchronize();
  end = std::chrono::high_resolution_clock::now();
  duration = std::chrono::duration_cast<std::chrono::microseconds>(end - start);
  printf("gpu takes %lld time for %d size arrays\n", (long long)duration.count(), n);

  // put d_c in h_c
  start = std::chrono::high_resolution_clock::now();
  cudaMemcpy(h_c, d_c, n*sizeof(float), cudaMemcpyDeviceToHost);
  end = std::chrono::high_resolution_clock::now();
  duration = std::chrono::duration_cast<std::chrono::microseconds>(end - start);
  printf("duration to copy back %d values to cpu is %lld\n", n, (long long)duration.count());

  // validate
  //for(int i = 0;i < 10;i++){
  // printf("c[%d] = %f\n",i,h_c[i]);
  //}

  start = std::chrono::high_resolution_clock::now();
  for(int i = 0; i < n ;i++){
    h_c[i] = h_a[i] + h_b[i];
  }
  end = std::chrono::high_resolution_clock::now();
  duration = std::chrono::duration_cast<std::chrono::microseconds>(end - start);
  printf("duration for cpu to add %d values is %lld", n, (long long)duration.count());
}




Overwriting main.cu


In [4]:
!nvcc main.cu -o main && ./main

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
copying 2 1000 size values 44
gpu takes 217 time for 1000 size arrays
duration to copy back 1000 values to cpu is 21
duration for cpu to add 1000 values is 4

In [40]:
%%writefile main.cu

#include "kernel.cu"
#include <stdio.h>
#include <chrono>

int main() {
  // declare in cpu
  int n = 100000;
  float h_a[n];
  float h_b[n];
  float h_c[n];
  for(int i = 0; i < n ;i++){
    h_a[i] = i;
    h_b[i] = n-i;
    h_c[i] = 0;
  }

  // declare for gpu
  float* d_a;
  cudaMalloc(&d_a, n * sizeof(float));
  float* d_b;
  cudaMalloc(&d_b, n * sizeof(float));
  float* d_c;
  cudaMalloc(&d_c, n * sizeof(float));

  // put cpu values in gpu
  auto start = std::chrono::high_resolution_clock::now();
  cudaMemcpy(d_a, h_a, n*sizeof(float), cudaMemcpyHostToDevice);
  cudaMemcpy(d_b, h_b, n*sizeof(float), cudaMemcpyHostToDevice);
  auto end = std::chrono::high_resolution_clock::now();
  auto duration = std::chrono::duration_cast<std::chrono::microseconds>(end - start);
  printf("copying 2 %d size values %lld\n", n, (long long)duration.count());


  // call kernel
  int threadPerBlock = 256;
  int numBlocks = ((n+threadPerBlock-1)/threadPerBlock);
  // timing
  start = std::chrono::high_resolution_clock::now();
  myKernel<<<numBlocks, threadPerBlock>>>(d_a, d_b, d_c, n);
  cudaDeviceSynchronize();
  end = std::chrono::high_resolution_clock::now();
  duration = std::chrono::duration_cast<std::chrono::microseconds>(end - start);
  printf("gpu takes %lld time for %d size arrays\n", (long long)duration.count(), n);

  // put d_c in h_c
  start = std::chrono::high_resolution_clock::now();
  cudaMemcpy(h_c, d_c, n*sizeof(float), cudaMemcpyDeviceToHost);
  end = std::chrono::high_resolution_clock::now();
  duration = std::chrono::duration_cast<std::chrono::microseconds>(end - start);
  printf("duration to copy back %d values to cpu is %lld\n", n, (long long)duration.count());

  // validate
  //for(int i = 0;i < 10;i++){
  // printf("c[%d] = %f\n",i,h_c[i]);
  //}

  start = std::chrono::high_resolution_clock::now();
  for(int i = 0; i < n ;i++){
    h_c[i] = h_a[i] + h_b[i];
  }
  end = std::chrono::high_resolution_clock::now();
  duration = std::chrono::duration_cast<std::chrono::microseconds>(end - start);
  printf("duration for cpu to add %d values is %lld", n, (long long)duration.count());
}




Overwriting main.cu


In [41]:
!nvcc main.cu -o main && ./main

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
copying 2 100000 size values 249
gpu takes 208 time for 100000 size arrays
duration to copy back 100000 values to cpu is 127
duration for cpu to add 100000 values is 274

In [42]:
%%writefile main.cu

#include "kernel.cu"
#include <stdio.h>
#include <chrono>

int main() {
  // declare in cpu
  int n = 1000000;
  float h_a[n];
  float h_b[n];
  float h_c[n];
  for(int i = 0; i < n ;i++){
    h_a[i] = i;
    h_b[i] = n-i;
    h_c[i] = 0;
  }

  // declare for gpu
  float* d_a;
  cudaMalloc(&d_a, n * sizeof(float));
  float* d_b;
  cudaMalloc(&d_b, n * sizeof(float));
  float* d_c;
  cudaMalloc(&d_c, n * sizeof(float));

  // put cpu values in gpu
  auto start = std::chrono::high_resolution_clock::now();
  cudaMemcpy(d_a, h_a, n*sizeof(float), cudaMemcpyHostToDevice);
  cudaMemcpy(d_b, h_b, n*sizeof(float), cudaMemcpyHostToDevice);
  auto end = std::chrono::high_resolution_clock::now();
  auto duration = std::chrono::duration_cast<std::chrono::microseconds>(end - start);
  printf("copying 2 %d size values %lld\n", n, (long long)duration.count());


  // call kernel
  int threadPerBlock = 256;
  int numBlocks = ((n+threadPerBlock-1)/threadPerBlock);
  // timing
  start = std::chrono::high_resolution_clock::now();
  myKernel<<<numBlocks, threadPerBlock>>>(d_a, d_b, d_c, n);
  cudaDeviceSynchronize();
  end = std::chrono::high_resolution_clock::now();
  duration = std::chrono::duration_cast<std::chrono::microseconds>(end - start);
  printf("gpu takes %lld time for %d size arrays\n", (long long)duration.count(), n);

  // put d_c in h_c
  start = std::chrono::high_resolution_clock::now();
  cudaMemcpy(h_c, d_c, n*sizeof(float), cudaMemcpyDeviceToHost);
  end = std::chrono::high_resolution_clock::now();
  duration = std::chrono::duration_cast<std::chrono::microseconds>(end - start);
  printf("duration to copy back %d values to cpu is %lld\n", n, (long long)duration.count());

  // validate
  //for(int i = 0;i < 10;i++){
  // printf("c[%d] = %f\n",i,h_c[i]);
  //}

  start = std::chrono::high_resolution_clock::now();
  for(int i = 0; i < n ;i++){
    h_c[i] = h_a[i] + h_b[i];
  }
  end = std::chrono::high_resolution_clock::now();
  duration = std::chrono::duration_cast<std::chrono::microseconds>(end - start);
  printf("duration for cpu to add %d values is %lld", n, (long long)duration.count());
}




Overwriting main.cu


In [ ]:
!nvcc main.cu -o main && ./main

In [15]:
%%writefile main.cu

#include "kernel.cu"
#include <stdio.h>
#include <chrono>

int main() {
  // declare in cpu
  int n = 10000000;
  float *h_a = new float[n];
  float *h_b = new float[n];
  float *h_c = new float[n];
  for(int i = 0; i < n ;i++){
    h_a[i] = i;
    h_b[i] = n-i;
    h_c[i] = 0;
  }

  // declare for gpu
  float* d_a;
  cudaMalloc(&d_a, n * sizeof(float));
  float* d_b;
  cudaMalloc(&d_b, n * sizeof(float));
  float* d_c;
  cudaMalloc(&d_c, n * sizeof(float));

  auto start2 = std::chrono::high_resolution_clock::now();
  // put cpu values in gpu
  auto start = std::chrono::high_resolution_clock::now();
  cudaMemcpy(d_a, h_a, n*sizeof(float), cudaMemcpyHostToDevice);
  cudaMemcpy(d_b, h_b, n*sizeof(float), cudaMemcpyHostToDevice);
  auto end = std::chrono::high_resolution_clock::now();
  auto duration = std::chrono::duration_cast<std::chrono::microseconds>(end - start);
  printf("copying 2 %d size values %lld\n", n, (long long)duration.count());


  // call kernel
  int threadPerBlock = 256;
  int numBlocks = ((n+threadPerBlock-1)/threadPerBlock);
  // timing
  start = std::chrono::high_resolution_clock::now();
  myKernel<<<numBlocks, threadPerBlock>>>(d_a, d_b, d_c, n);
  cudaDeviceSynchronize();
  end = std::chrono::high_resolution_clock::now();
  duration = std::chrono::duration_cast<std::chrono::microseconds>(end - start);
  printf("gpu takes %lld time for %d size arrays\n", (long long)duration.count(), n);


  // put d_c in h_c
  start = std::chrono::high_resolution_clock::now();
  cudaMemcpy(h_c, d_c, n*sizeof(float), cudaMemcpyDeviceToHost);
  end = std::chrono::high_resolution_clock::now();
  duration = std::chrono::duration_cast<std::chrono::microseconds>(end - start);
  auto end2 = std::chrono::high_resolution_clock::now();
  auto duration2 = std::chrono::duration_cast<std::chrono::microseconds>(end2 - start2);


  printf("duration to copy back %d values to cpu is %lld\n", n, (long long)duration.count());

  // validate
  //for(int i = 0;i < 10;i++){
  // printf("c[%d] = %f\n",i,h_c[i]);
  //}

  start = std::chrono::high_resolution_clock::now();
  for(int i = 0; i < n ;i++){
    h_c[i] = h_a[i] + h_b[i];
  }
  end = std::chrono::high_resolution_clock::now();
  duration = std::chrono::duration_cast<std::chrono::microseconds>(end - start);
  printf("duration for cpu to add %d values is %lld\n", n, (long long)duration.count());
  printf("duration for gpu to add %d values is %lld\n", n, (long long)duration2.count());
  printf("cpu is %f times gpu", (float)duration.count()/(float)duration2.count());
}




Overwriting main.cu


In [16]:
!nvcc main.cu -o main && ./main

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
copying 2 10000000 size values 17124
gpu takes 645 time for 10000000 size arrays
duration to copy back 10000000 values to cpu is 8593
duration for cpu to add 10000000 values is 31092
duration for gpu to add 10000000 values is 26428
cpu is 1.176479 times gpu

In [10]:
!nvcc kernel.cu -o kernel && ./kernel

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
/usr/bin/ld: /usr/lib/gcc/x86_64-linux-gnu/11/../../../x86_64-linux-gnu/Scrt1.o: in function `_start':
(.text+0x1b): undefined reference to `main'
collect2: error: ld returned 1 exit status
